# Lezione 5 — Apprendimento strutturato e avversariale con TensorFlow

**Obiettivi della lezione**

- Spiegare cosa porta il framework Neural Structured Learning rispetto all'addestramento standard
- Implementare un modello avversariale e un modello robusto con NSL
- Comprendere il ruolo della regolarizzazione di adiacenza e della perturbazione nel migliorare la robustezza

Neural Structured Learning (NSL) è un framework di TensorFlow che aggiunge all'addestramento standard la capacità di sfruttare la struttura nei dati: i dati non sono isolati, ma connessi a dati simili attraverso un grafo di adiacenza.

## 1. Cosa porta NSL rispetto all'addestramento standard?

Nell'addestramento standard, la funzione di perdita (loss) confronta solo la predizione del modello con l'etichetta reale di ogni esempio singolo. Ignora completamente la relazione tra esempi simili.

Con NSL, il modello impara anche a mantenere predizioni simili per esempi adiacenti (simili) nel grafo. Questo è utile quando:

- I dati hanno una struttura naturale (ad es. una rete di utenti in una piattaforma sociale)
- Vuoi che il modello sia più robusto a piccoli cambiamenti negli input (robustezza avversariale)
- Hai pochi dati etichettati e vuoi sfruttare i dati non etichettati

## 2. Installazione

`neural_structured_learning` non è installato di default su Colab, quindi lo installiamo prima.

In [ ]:
# Instala Neural Structured Learning (NSL) si no está disponible
!pip install --quiet neural-structured-learning


In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import neural_structured_learning as nsl  # convención habitual: abreviar a "nsl"
import matplotlib.pyplot as plt

print('TensorFlow:', tf.__version__)


## 3. Grafi naturali: il dataset Cora

Un **grafo naturale** usa relazioni che già esistono nei dati: documento citato da quale altro documento, persona che segue quale altra persona, ecc.

Il dataset **Cora** è una rete di 2.708 pubblicazioni scientifiche (nodi) collegate da 5.429 citazioni (archi). Ogni nodo ha 1.433 caratteristiche (le parole che compaiono nel riassunto). Il compito: classificare ogni documento in una di 7 categorie (e.g., reti neurali, machine learning, linguistica).

In [ ]:
# Descarga el dataset Cora (grafo de citas) y el script de preprocesado de NSL
!wget --quiet -P /tmp https://linqs-data.soe.ucsc.edu/public/lbc/cora.tgz
!tar -C /tmp -xzf /tmp/cora.tgz
!wget --quiet https://raw.githubusercontent.com/tensorflow/neural-structured-learning/master/neural_structured_learning/examples/preprocess/cora/preprocess_cora_dataset.py

# El script genera los ejemplos de entrenamiento ya aumentados con sus vecinos del grafo
!python preprocess_cora_dataset.py \
  --input_cora_content=/tmp/cora/cora.content \
  --input_cora_graph=/tmp/cora/cora.cites \
  --max_nbrs=5 \
  --output_train_data=/tmp/cora/train_merged_examples.tfr \
  --output_test_data=/tmp/cora/test_examples.tfr


In [ ]:
NBR_FEATURE_PREFIX = 'NL_nbr_'
NBR_WEIGHT_SUFFIX = '_weight'

class HParamsGraph(object):
    def __init__(self):
        self.num_classes = 7
        self.max_seq_length = 1433          # tamaño del vocabulario (bag-of-words)
        self.distance_type = nsl.configs.DistanceType.L2
        self.graph_regularization_multiplier = 0.1  # peso del término de grafo en la pérdida
        self.num_neighbors = 1
        self.num_fc_units = [50, 50]
        self.train_epochs = 20               # en el tutorial original: 100 (aquí lo reducimos para ir más rápido en clase)
        self.batch_size = 128
        self.dropout_rate = 0.5

HPARAMS = HParamsGraph()

def make_dataset(file_path, training=False):
    def parse_example(example_proto):
        feature_spec = {
            'words': tf.io.FixedLenFeature([HPARAMS.max_seq_length], tf.int64,
                                            default_value=tf.constant(0, dtype=tf.int64, shape=[HPARAMS.max_seq_length])),
            'label': tf.io.FixedLenFeature((), tf.int64, default_value=-1),
        }
        if training:
            for i in range(HPARAMS.num_neighbors):
                nbr_feature_key = '{}{}_{}'.format(NBR_FEATURE_PREFIX, i, 'words')
                nbr_weight_key = '{}{}{}'.format(NBR_FEATURE_PREFIX, i, NBR_WEIGHT_SUFFIX)
                feature_spec[nbr_feature_key] = tf.io.FixedLenFeature(
                    [HPARAMS.max_seq_length], tf.int64,
                    default_value=tf.constant(0, dtype=tf.int64, shape=[HPARAMS.max_seq_length]))
                feature_spec[nbr_weight_key] = tf.io.FixedLenFeature([1], tf.float32, default_value=tf.constant([0.0]))
        features = tf.io.parse_single_example(example_proto, feature_spec)
        label = features.pop('label')
        return features, label

    dataset = tf.data.TFRecordDataset([file_path])
    if training:
        dataset = dataset.shuffle(10000)
    dataset = dataset.map(parse_example)
    dataset = dataset.batch(HPARAMS.batch_size)
    return dataset

train_dataset = make_dataset('/tmp/cora/train_merged_examples.tfr', training=True)
test_dataset = make_dataset('/tmp/cora/test_examples.tfr')


In [ ]:
def make_mlp_functional_model(hparams):
    """Perceptron multicapa (MLP) sencillo: el modelo base de siempre, sin nada especial de NSL."""
    inputs = tf.keras.Input(shape=(hparams.max_seq_length,), dtype='int64', name='words')
    cur_layer = tf.keras.layers.Lambda(lambda x: tf.keras.backend.cast(x, tf.float32))(inputs)
    for num_units in hparams.num_fc_units:
        cur_layer = tf.keras.layers.Dense(num_units, activation='relu')(cur_layer)
        cur_layer = tf.keras.layers.Dropout(hparams.dropout_rate)(cur_layer)
    outputs = tf.keras.layers.Dense(hparams.num_classes)(cur_layer)
    return tf.keras.Model(inputs, outputs=outputs)

# --- Modelo base, SIN regularización de grafos ---
base_model = make_mlp_functional_model(HPARAMS)
base_model.compile(optimizer='adam',
                    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                    metrics=['accuracy'])
base_model.fit(train_dataset, epochs=HPARAMS.train_epochs, verbose=0)
base_eval = dict(zip(base_model.metrics_names, base_model.evaluate(test_dataset, verbose=0)))
print('Accuracy modelo base (sin grafo):', base_eval['accuracy'])


In [ ]:
# --- Mismo modelo, envuelto con GraphRegularization ---
graph_reg_config = nsl.configs.make_graph_reg_config(
    max_neighbors=HPARAMS.num_neighbors,
    multiplier=HPARAMS.graph_regularization_multiplier,
    distance_type=HPARAMS.distance_type,
    sum_over_axis=-1)

graph_model = nsl.keras.GraphRegularization(make_mlp_functional_model(HPARAMS), graph_reg_config)
graph_model.compile(optimizer='adam',
                     loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                     metrics=['accuracy'])
graph_model.fit(train_dataset, epochs=HPARAMS.train_epochs, verbose=0)
graph_eval = dict(zip(graph_model.metrics_names, graph_model.evaluate(test_dataset, verbose=0)))
print('Accuracy modelo con regularización de grafos:', graph_eval['accuracy'])


Il modello con regolarizzazione di grafi di solito supera il modello base perché sfrutta il fatto che due documenti citati l'uno dall'altro tendono ad appartenere alla stessa categoria.

## 4. Grafi sintetizzati: quando non hai un grafo di serie

La maggior parte dei dataset non ha un grafo naturale. In questo caso, NSL sintetizza un grafo in base alla somiglianza: se due esempi hanno caratteristiche simili, vengono connessi.

In [ ]:
# Simulamos 6 "documentos" ya convertidos a embeddings (en un caso real vendrían
# de un modelo preentrenado, p. ej. Swivel o BERT, aplicado sobre el texto real)
np.random.seed(0)
embeddings = {
    'doc_0': np.array([0.9, 0.1, 0.0]),
    'doc_1': np.array([0.85, 0.15, 0.0]),   # parecido a doc_0
    'doc_2': np.array([0.0, 0.9, 0.1]),
    'doc_3': np.array([0.05, 0.92, 0.1]),   # parecido a doc_2
    'doc_4': np.array([0.1, 0.0, 0.9]),
    'doc_5': np.array([0.0, 0.1, 0.88]),    # parecido a doc_4
}

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarity_threshold = 0.99  # el mismo valor que usa el tutorial oficial de IMDb
edges = []
names = list(embeddings.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        sim = cosine_similarity(embeddings[names[i]], embeddings[names[j]])
        if sim >= similarity_threshold:
            edges.append((names[i], names[j], round(float(sim), 4)))

print('Grafo sintetizado (aristas por encima del umbral de similitud):')
for edge in edges:
    print(' ', edge)


Nota che il grafo risultante collega `doc_0` con `doc_1`, `doc_2` con `doc_3` (i loro embedding sono molto simili), ma non collega `doc_0` con `doc_3` (la loro somiglianza è al di sotto della soglia).

## 5. Regolarizzazione avversariale: robustezza contro gli attacchi

Un **esempio avversariale** è un input leggermente modificato che inganna il modello, anche se l'occhio umano non vede differenza. Una perturbazione imperquettibile in un'immagine potrebbe far classificare un gatto come un cane.

NSL aggiunge una regolarizzazione avversariale: durante l'addestramento, il modello allena non solo sull'esempio originale ma anche su versioni perturbate, imparando così a essere robusto contro piccole modifiche.

In [ ]:
class HParamsAdv(object):
    def __init__(self):
        self.input_shape = [28, 28, 1]
        self.num_classes = 10
        self.conv_filters = [32, 64, 64]
        self.kernel_size = (3, 3)
        self.pool_size = (2, 2)
        self.num_fc_units = [64]
        self.batch_size = 32
        self.epochs = 3                # en el tutorial original: 5 (reducido para ir más rápido en clase)
        self.adv_multiplier = 0.2      # peso del término adversarial en la pérdida
        self.adv_step_size = 0.2       # magnitud de la perturbación
        self.adv_grad_norm = 'infinity'

HPARAMS_ADV = HParamsAdv()

IMAGE_INPUT_NAME = 'image'
LABEL_INPUT_NAME = 'label'

datasets = tfds.load('mnist')
train_dataset_adv = datasets['train']
test_dataset_adv = datasets['test']

def normalize(features):
    features[IMAGE_INPUT_NAME] = tf.cast(features[IMAGE_INPUT_NAME], tf.float32) / 255.0
    return features

def convert_to_tuples(features):
    return features[IMAGE_INPUT_NAME], features[LABEL_INPUT_NAME]

def convert_to_dictionaries(image, label):
    return {IMAGE_INPUT_NAME: image, LABEL_INPUT_NAME: label}

train_dataset_adv = train_dataset_adv.map(normalize).shuffle(10000).batch(HPARAMS_ADV.batch_size).map(convert_to_tuples)
test_dataset_adv = test_dataset_adv.map(normalize).batch(HPARAMS_ADV.batch_size).map(convert_to_tuples)


In [ ]:
def build_base_model(hparams):
    """CNN sencilla: 3 capas convolucionales + 2 densas, con la Functional API de Keras."""
    inputs = tf.keras.Input(shape=hparams.input_shape, dtype=tf.float32, name=IMAGE_INPUT_NAME)
    x = inputs
    for i, num_filters in enumerate(hparams.conv_filters):
        x = tf.keras.layers.Conv2D(num_filters, hparams.kernel_size, activation='relu')(x)
        if i < len(hparams.conv_filters) - 1:
            x = tf.keras.layers.MaxPooling2D(hparams.pool_size)(x)
    x = tf.keras.layers.Flatten()(x)
    for num_units in hparams.num_fc_units:
        x = tf.keras.layers.Dense(num_units, activation='relu')(x)
    pred = tf.keras.layers.Dense(hparams.num_classes)(x)
    return tf.keras.Model(inputs=inputs, outputs=pred)

# --- Modelo base, SIN regularización adversarial ---
base_model_mnist = build_base_model(HPARAMS_ADV)
base_model_mnist.compile(optimizer='adam',
                          loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                          metrics=['acc'])
base_model_mnist.fit(train_dataset_adv, epochs=HPARAMS_ADV.epochs, verbose=0)
base_results = dict(zip(base_model_mnist.metrics_names, base_model_mnist.evaluate(test_dataset_adv, verbose=0)))
print('Accuracy modelo base sobre datos limpios:', base_results['acc'])


In [ ]:
# --- Mismo modelo, envuelto con AdversarialRegularization ---
adv_config = nsl.configs.make_adv_reg_config(
    multiplier=HPARAMS_ADV.adv_multiplier,
    adv_step_size=HPARAMS_ADV.adv_step_size,
    adv_grad_norm=HPARAMS_ADV.adv_grad_norm)

base_adv_model = build_base_model(HPARAMS_ADV)
adv_model = nsl.keras.AdversarialRegularization(
    base_adv_model, label_keys=[LABEL_INPUT_NAME], adv_config=adv_config)

# AdversarialRegularization necesita la etiqueta junto a la imagen, así que
# convertimos el dataset de tuplas (image, label) a diccionarios {'image':..., 'label':...}
train_set_for_adv_model = train_dataset_adv.map(convert_to_dictionaries)
test_set_for_adv_model = test_dataset_adv.map(convert_to_dictionaries)

adv_model.compile(optimizer='adam',
                   loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                   metrics=['acc'])
adv_model.fit(train_set_for_adv_model, epochs=HPARAMS_ADV.epochs, verbose=0)
adv_results = dict(zip(adv_model.metrics_names, adv_model.evaluate(test_set_for_adv_model, verbose=0)))
print('Accuracy modelo adversarial-regularizado sobre datos limpios:', adv_results['sparse_categorical_accuracy'])


### Robustezza contro esempi perturbati

Ora generiamo esempi avversariali intenzionalmente e li testiamo su un modello standard (senza NSL) e su uno robusto (con regolarizzazione avversariale).

In [ ]:
# Usamos el modelo base para generar las perturbaciones (referencia)
reference_model = nsl.keras.AdversarialRegularization(
    base_model_mnist, label_keys=[LABEL_INPUT_NAME], adv_config=adv_config)
reference_model.compile(optimizer='adam',
                         loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                         metrics=['acc'])

models_to_eval = {
    'base': base_model_mnist,
    'adv-regularized': adv_model.base_model,
}
metrics = {name: tf.keras.metrics.SparseCategoricalAccuracy() for name in models_to_eval}

for batch in test_set_for_adv_model:
    perturbed_batch = reference_model.perturb_on_batch(batch)
    perturbed_batch[IMAGE_INPUT_NAME] = tf.clip_by_value(perturbed_batch[IMAGE_INPUT_NAME], 0.0, 1.0)
    y_true = perturbed_batch.pop(LABEL_INPUT_NAME)
    for name, model in models_to_eval.items():
        y_pred = model(perturbed_batch)
        metrics[name](y_true, y_pred)

for name, metric in metrics.items():
    print('Accuracy del modelo %s frente a imágenes perturbadas: %f' % (name, metric.result().numpy()))


Dovresti vedere che il modello base perde molta precisione di fronte alle immagini perturbate, mentre il modello robusto mantiene una precisione molto più alta. Ecco il valore della regolarizzazione avversariale.

## Riepilogo della lezione

- NSL aggiunge un termine di regolarizzazione alla perdita standard, sfruttando la struttura nei dati
- Funziona sia con grafi naturali (relazioni che già esistono) che con grafi sintetizzati (basati sulla somiglianza)
- La regolarizzazione avversariale prepara il modello ad affrontare input leggermente modificati
- Un modello robusto è più adatto alla distribuzione perché gestisce meglio le variazioni del mondo reale